# Project 1 ANSC 4040 Model Tuning NEW

This is a notebook for Project 1 that will be used for expanding the same size and tuning the hyperparameters of original model

In [1]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

Original dataset
       │
       ├── 70% TRAIN ───────────────┐
       │                            │
       │                      Hyperparameter
       │                         tuning
       │                            │
       │                            ↓
       │                       Best RF model
       │                            │
       ├── 20% VALIDATION ──────────┤
       │                            ↓
       │                     Model evaluation
       │
       └── 10% TEST ────────────────→ FINAL evaluation

In [ ]:
# Load Data Sets 
train_path = "train_data.csv"
val_path = "val_data.csv"
test_path = "test_data.csv"

In [ ]:
# Get Animal IDs from each split

train_ids = set()

for chunk in pd.read_csv(
    train_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000
):
    train_ids.update(
        chunk["AnimalId"].dropna().unique()
    )


val_ids = set()

for chunk in pd.read_csv(
    val_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000
):
    val_ids.update(
        chunk["AnimalId"].dropna().unique()
    )


test_ids = set()

for chunk in pd.read_csv(
    test_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000
):
    test_ids.update(
        chunk["AnimalId"].dropna().unique()
    )


# Cows represented in ALL THREE splits
shared_cows = sorted(
    train_ids & val_ids & test_ids
)

print(f"Cows in all three splits: {len(shared_cows):,}")

In [ ]:
#at least 400 cows
if len(shared_cows) < 400:
    raise ValueError(
        f"Only {len(shared_cows)} cows occur in all three splits."
    )

In [ ]:
#select cows
selected_cows = set(
    pd.Series(
        shared_cows,
        dtype="string"
    ).sample(
        n=400,
        random_state=42
    ).tolist()
)

print(f"Selected cows: {len(selected_cows)}")

In [ ]:
#load cows

train_400 = load_selected_cows(
    train_path,
    selected_cows
)

val_400 = load_selected_cows(
    val_path,
    selected_cows
)

test_400 = load_selected_cows(
    test_path,
    selected_cows
)

print("TRAIN")
print(f"Rows: {len(train_400):,}")
print(f"Cows: {train_400['AnimalId'].nunique()}")

print("\nVALIDATION")
print(f"Rows: {len(val_400):,}")
print(f"Cows: {val_400['AnimalId'].nunique()}")

print("\nTEST")
print(f"Rows: {len(test_400):,}")
print(f"Cows: {test_400['AnimalId'].nunique()}")

# check 400 in each set 

In [ ]:
#create model features and date variables
feature_cols = [
    "Avgmilkflow",
    "EventYear",
    "EventMonth",
    "EventDay",
    "EventDayOfWeek",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking"
]

for data in [train_400, val_400, test_400]:

    event_date = pd.to_datetime(
        data["EventDate"]
    )

    data["EventYear"] = event_date.dt.year
    data["EventMonth"] = event_date.dt.month
    data["EventDay"] = event_date.dt.day
    data["EventDayOfWeek"] = event_date.dt.dayofweek

In [ ]:
#create x and y for training validation and test data

X_train = train_400[feature_cols].astype("float32")
y_train = train_400["AnimalId"]

X_val = val_400[feature_cols].astype("float32")
y_val = val_400["AnimalId"]

X_test = test_400[feature_cols].astype("float32")
y_test = test_400["AnimalId"]

# TUNE ONLY training data

In [ ]:
#3-fold stratified CV 

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

param_distributions = {
    "n_estimators": [50, 75, 100, 150],
    "max_depth": [8, 10, 15, 20, 25, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2"]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="accuracy",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=2,
    return_train_score=False
)

random_search.fit(
    X_train,
    y_train
)

print("Best parameters:")
print(random_search.best_params_)

print(
    f"\nBest cross-validation accuracy: "
    f"{random_search.best_score_:.2%}"
)

In [ ]:
#evaluate best model on 20% validation set

best_rf = random_search.best_estimator_

y_val_pred = best_rf.predict(X_val)

val_top1 = accuracy_score(
    y_val,
    y_val_pred
)

print(
    f"Validation Top-1 accuracy: "
    f"{val_top1:.2%}"
)

In [ ]:
#calculate top 5 and 10 accuracy for validation set
y_val_proba = best_rf.predict_proba(X_val)

classes = best_rf.classes_

top_10_indices = np.argsort(
    y_val_proba,
    axis=1
)[:, -10:][:, ::-1]

top_10_predictions = classes[top_10_indices]

y_true = np.asarray(y_val)

val_top5 = np.mean(
    np.any(
        top_10_predictions[:, :5]
        == y_true[:, None],
        axis=1
    )
)

val_top10 = np.mean(
    np.any(
        top_10_predictions[:, :10]
        == y_true[:, None],
        axis=1
    )
)

print(f"Validation Top-1:  {val_top1:.2%}")
print(f"Validation Top-5:  {val_top5:.2%}")
print(f"Validation Top-10: {val_top10:.2%}")